# Runner A: Phase 2 RAGAS untuk Phase 1 yang Belum/Perlu Re-evaluasi (GPT-4.1-mini)

Notebook ini menjalankan evaluasi RAGAS (4 metrik) untuk konfigurasi yang phase 2 RAGAS-nya belum dijalankan ATAU perlu di-rerun karena ada perubahan Phase 1.

**Konfigurasi target (6 total):**

Kelompok 2 (Hybrid tanpa Ekspansi Akronim) — semua 4 belum ada phase 2:
- `baseline` — Hybrid retrieval saja
- `qr` — Hybrid + Query Rewriting (Versi A)
- `cr` — Hybrid + Context Reranking
- `qr_cr` — Hybrid + QR + CR

Kelompok 3 (Hybrid dengan Ekspansi Akronim) — 2 perlu re-run karena QR di-patch ke Versi A:
- `qr` — Hybrid + Ekspansi + QR (Versi A)
- `qr_cr` — Hybrid + Ekspansi + QR + CR (Versi A)

**Output:** `results/<kelompok>/{config}_phase2_custom.json`

Resume-aware: kalau sampel sudah dievaluasi, akan di-skip.

## 1. Setup & Imports

In [10]:
import os
import sys
import json
import time
from datetime import datetime, timedelta
from pathlib import Path
from openai import OpenAI

# Progress bar (tqdm auto-detect notebook vs CLI)
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# Resolve PROJECT_ROOT
_HERE = Path('.').resolve()
PROJECT_ROOT = next((p for p in [_HERE] + list(_HERE.parents) if p.name == 'Code TA'), _HERE.parent.parent.parent)
NOTEBOOKS_V2 = PROJECT_ROOT / 'notebooks'
SCRIPTS_DIR    = NOTEBOOKS_V2 / 'scripts'
RESULTS_BASE   = PROJECT_ROOT / 'results'

sys.path.insert(0, str(SCRIPTS_DIR))
from ragas_evaluator import evaluate_sample

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RESULTS_BASE : {RESULTS_BASE}')
print()
for k in ['20_hybrid_tanpa_ekspansi', '30_hybrid_dengan_ekspansi']:
    d = RESULTS_BASE / k
    print(f'[{k}]  files:')
    for p in sorted(d.glob('*.json')):
        print(f'  - {p.name}')

PROJECT_ROOT : C:\Users\Ricky Wijaya\Documents\STI\Semester 8\TA\Code TA
RESULTS_BASE : C:\Users\Ricky Wijaya\Documents\STI\Semester 8\TA\Code TA\results

[20_hybrid_tanpa_ekspansi]  files:
  - baseline_phase1_answers.json
  - baseline_phase2_custom.json
  - baseline_retrieval_metrics.json
  - cr_phase1_answers.json
  - cr_phase2_custom.json
  - cr_retrieval_metrics.json
  - qr_cr_phase1_answers.json
  - qr_cr_phase2_custom.json
  - qr_cr_retrieval_metrics.json
  - qr_phase1_answers.json
  - qr_phase2_custom.json
  - qr_retrieval_metrics.json
[30_hybrid_dengan_ekspansi]  files:
  - baseline_phase1_answers.json
  - baseline_phase2_custom.json
  - baseline_retrieval_metrics.json
  - cr_phase1_answers.json
  - cr_phase2_custom.json
  - cr_retrieval_metrics.json
  - qr_phase2_custom.json
  - sh_qr_cr_openai_phase1_answers.json
  - sh_qr_openai_phase1_answers.json


## 2. OpenAI client + generator wrapper

In [9]:
# Set env var
# os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_KEY_HERE')
EVAL_MODEL = 'gpt-4.1-mini'  # model untuk RAGAS judge

client = OpenAI(api_key=OPENAI_API_KEY)


# Counter untuk track API calls
_api_call_count = [0]


def openai_generate(prompt: str, max_tokens: int = 300, temperature: float = 0.0) -> str:
    """Wrapper standar untuk pemanggilan OpenAI Chat Completion."""
    _api_call_count[0] += 1
    resp = client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def reset_api_counter():
    _api_call_count[0] = 0


def get_api_count():
    return _api_call_count[0]


# Smoke test
reset_api_counter()
print(openai_generate('Reply with: ok', max_tokens=5))
print(f'API calls so far: {get_api_count()}')

ok
API calls so far: 1


## 3. Konfigurasi target + helper fungsi dengan progress tracking

In [3]:
# Tuple (kelompok, config) — 6 target
CONFIGS = [
    ('20_hybrid_tanpa_ekspansi', 'baseline'),
    ('20_hybrid_tanpa_ekspansi', 'qr'),
    ('20_hybrid_tanpa_ekspansi', 'cr'),
    ('20_hybrid_tanpa_ekspansi', 'qr_cr'),
    ('30_hybrid_dengan_ekspansi', 'qr'),
    ('30_hybrid_dengan_ekspansi', 'qr_cr'),
]
REQUIRED_METRICS = ['faithfulness', 'context_recall', 'answer_relevancy', 'context_precision']


def format_eta(seconds):
    """Format detik jadi human-readable."""
    if seconds < 60:
        return f'{seconds:.0f}s'
    if seconds < 3600:
        return f'{seconds/60:.1f}m'
    return f'{seconds/3600:.1f}h'


def run_phase2_for_config(kelompok: str, config_name: str, max_samples: int = 500, config_idx: int = 0, total_configs: int = 6):
    results_dir = RESULTS_BASE / kelompok
    phase1_path = results_dir / f'{config_name}_phase1_answers.json'
    phase2_path = results_dir / f'{config_name}_phase2_custom.json'
    label = f'{kelompok}/{config_name}'

    if not phase1_path.exists():
        print(f'[SKIP] {label}: phase1 file tidak ada')
        return None

    with open(phase1_path, 'r', encoding='utf-8') as f:
        phase1_data = json.load(f)

    samples = phase1_data.get('results', phase1_data.get('answers', []))[:max_samples]

    print()
    print('=' * 80)
    print(f'CONFIG {config_idx + 1}/{total_configs} : {label}  ({datetime.now().strftime("%H:%M:%S")})')
    print('=' * 80)
    print(f'  Phase1 samples loaded: {len(samples)}')

    # Load existing phase2 untuk resume
    if phase2_path.exists():
        with open(phase2_path, 'r', encoding='utf-8') as f:
            p2 = json.load(f)
        done = {r['idx'] for r in p2.get('results', []) if all(m in r for m in REQUIRED_METRICS)}
        results = p2.get('results', [])
        print(f'  Resume mode  : {len(done)}/{len(samples)} sampel sudah lengkap')
    else:
        done, results = set(), []
        print(f'  Fresh start  : {len(samples)} sampel akan diproses')

    todo = [s for s in samples if s.get('idx') not in done]
    n_todo = len(todo)
    print(f'  Perlu eval   : {n_todo} sampel')

    if n_todo == 0:
        print(f'  [DONE] Semua sampel sudah ter-evaluasi.')
        return phase2_path

    # Reset counter API
    api_start = get_api_count()
    cfg_start = time.time()
    errors = 0

    # Progress bar dengan tqdm
    pbar = tqdm(
        todo,
        desc=f'{label[-30:]:<30}',
        unit='sample',
        ncols=120,
        bar_format='{desc} |{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]'
    )

    for k, sample in enumerate(pbar, 1):
        try:
            metrics = evaluate_sample(openai_generate, sample)
        except Exception as e:
            errors += 1
            pbar.set_postfix(errors=errors, api=get_api_count() - api_start)
            print(f'\n  [ERROR] idx={sample.get("idx")}: {type(e).__name__}: {str(e)[:80]}')
            continue

        row = {
            'idx': sample.get('idx'),
            'ground_truth': sample.get('ground_truth') or sample.get('final_decision'),
            'predicted_label': sample.get('predicted_label'),
            'is_correct': sample.get('is_correct'),
            **metrics,
        }
        results.append(row)

        # Update progress bar postfix dengan info berguna
        api_calls = get_api_count() - api_start
        elapsed = time.time() - cfg_start
        avg_per_sample = elapsed / k
        pbar.set_postfix(
            api=api_calls,
            avg=f'{avg_per_sample:.1f}s',
            err=errors,
        )

        # Save inkremental tiap 25 sampel
        if k % 25 == 0 or k == n_todo:
            with open(phase2_path, 'w', encoding='utf-8') as f:
                json.dump({
                    'kelompok': kelompok,
                    'config': config_name,
                    'eval_model': EVAL_MODEL,
                    'metrics': REQUIRED_METRICS,
                    'results': results,
                }, f, indent=2)

    pbar.close()

    # Summary per-config
    cfg_elapsed = time.time() - cfg_start
    total_api = get_api_count() - api_start
    print()
    print(f'  Selesai dalam: {format_eta(cfg_elapsed)} ({cfg_elapsed:.0f}s)')
    print(f'  API calls    : {total_api} (avg {total_api/max(n_todo, 1):.1f}/sample)')
    print(f'  Errors       : {errors}')
    print(f'  Saved        : {phase2_path.name}')
    return phase2_path

## 4. Run untuk semua 6 konfigurasi

Setiap konfigurasi ~500 sampel × 4 metrik × ~3 LLM calls per metrik = **~6000 LLM calls per konfigurasi**.

Estimasi total: **3-5 jam** untuk 6 konfigurasi. Progress bar akan menampilkan ETA real-time.

In [4]:
# Reset counter global
reset_api_counter()

global_start = time.time()
print(f'\nMulai: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Total konfigurasi: {len(CONFIGS)} | Sampel per konfigurasi: 500')
print(f'Estimasi waktu: 3-5 jam total\n')

completed = []
for i, (kelompok, cfg) in enumerate(CONFIGS):
    result_path = run_phase2_for_config(kelompok, cfg, max_samples=500, config_idx=i, total_configs=len(CONFIGS))
    completed.append(((kelompok, cfg), result_path))

    # Update progress global setelah tiap config
    elapsed_total = time.time() - global_start
    remaining_configs = len(CONFIGS) - (i + 1)
    if i + 1 > 0:
        avg_per_config = elapsed_total / (i + 1)
        eta_total = avg_per_config * remaining_configs
        pct_done = ((i + 1) / len(CONFIGS)) * 100
        print()
        print(f'>>> Progress GLOBAL: {i+1}/{len(CONFIGS)} configs ({pct_done:.0f}%) | Elapsed: {format_eta(elapsed_total)} | ETA: {format_eta(eta_total)} <<<')

total_elapsed = time.time() - global_start
total_api = get_api_count()
print()
print('=' * 70)
print(f'[DONE] Semua {len(CONFIGS)} konfigurasi selesai dalam {format_eta(total_elapsed)} ({total_elapsed:.0f}s)')
print(f'Total API calls: {total_api}')
print(f'Selesai: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 70)


Mulai: 2026-06-04 20:12:52
Total konfigurasi: 6 | Sampel per konfigurasi: 500
Estimasi waktu: 3-5 jam total


CONFIG 1/6 : 20_hybrid_tanpa_ekspansi/baseline  (20:12:52)
  Phase1 samples loaded: 500
  Resume mode  : 500/500 sampel sudah lengkap
  Perlu eval   : 0 sampel
  [DONE] Semua sampel sudah ter-evaluasi.

>>> Progress GLOBAL: 1/6 configs (17%) | Elapsed: 0s | ETA: 0s <<<

CONFIG 2/6 : 20_hybrid_tanpa_ekspansi/qr  (20:12:52)
  Phase1 samples loaded: 500
  Resume mode  : 500/500 sampel sudah lengkap
  Perlu eval   : 0 sampel
  [DONE] Semua sampel sudah ter-evaluasi.

>>> Progress GLOBAL: 2/6 configs (33%) | Elapsed: 0s | ETA: 0s <<<

CONFIG 3/6 : 20_hybrid_tanpa_ekspansi/cr  (20:12:52)
  Phase1 samples loaded: 500
  Resume mode  : 500/500 sampel sudah lengkap
  Perlu eval   : 0 sampel
  [DONE] Semua sampel sudah ter-evaluasi.

>>> Progress GLOBAL: 3/6 configs (50%) | Elapsed: 0s | ETA: 0s <<<

CONFIG 4/6 : 20_hybrid_tanpa_ekspansi/qr_cr  (20:12:52)
  Phase1 samples loaded: 500
  R

## 5. Verifikasi hasil + ringkasan

In [5]:
from statistics import mean

print(f'{"Kelompok/Config":<45} {"N":>4} {"Acc":>7} {"Faith":>7} {"CRec":>7} {"ARel":>7} {"CPrec":>7}')
print('-' * 90)

for kelompok, cfg in CONFIGS:
    phase2 = RESULTS_BASE / kelompok / f'{cfg}_phase2_custom.json'
    label = f'{kelompok}/{cfg}'
    if not phase2.exists():
        print(f'{label:<45} (no phase2 file)')
        continue
    with open(phase2, 'r', encoding='utf-8') as f:
        r = json.load(f).get('results', [])
    if not r:
        continue
    n = len(r)
    acc = sum(1 for x in r if x.get('is_correct')) / n
    def safe(k):
        vals = [x.get(k) for x in r if isinstance(x.get(k), (int, float))]
        return mean(vals) if vals else 0.0
    print(f'{label:<45} {n:>4} {acc:>7.4f} {safe("faithfulness"):>7.4f} {safe("context_recall"):>7.4f} {safe("answer_relevancy"):>7.4f} {safe("context_precision"):>7.4f}')

Kelompok/Config                                  N     Acc   Faith    CRec    ARel   CPrec
------------------------------------------------------------------------------------------
20_hybrid_tanpa_ekspansi/baseline              500  0.6920  0.9805  0.0000  0.9822  0.4820
20_hybrid_tanpa_ekspansi/qr                    500  0.7080  0.9760  0.0000  0.9811  0.4656
20_hybrid_tanpa_ekspansi/cr                    500  0.6980  0.9810  0.0000  0.9810  0.4912
20_hybrid_tanpa_ekspansi/qr_cr                 500  0.7000  0.9823  0.0000  0.9819  0.4696
30_hybrid_dengan_ekspansi/qr                  (no phase2 file)
30_hybrid_dengan_ekspansi/qr_cr               (no phase2 file)


## 6. Perbaikan: Re-run Context Recall (Kel.2) + Full RAGAS (Kel.3 QR & QR+CR)

Sel di bawah memperbaiki dua masalah pada data hasil:

1. **Context Recall = 0 pada Kelompok 2.** Disebabkan bug pada `evaluate_sample` yang membaca `long_answer`/`ground_truth` (label "yes/no/maybe") alih-alih field `reference`. Bug sudah diperbaiki di `ragas_evaluator.py`. Bagian A **menghitung ulang hanya `context_recall`** untuk 4 konfigurasi Kel.2 dan menambal ke berkas phase2 yang ada (hemat API — metrik lain tidak dihitung ulang).

2. **RAGAS belum ada untuk Kel.3 QR & QR+CR.** Berkas phase1-nya bernama `sh_qr_openai_*` / `sh_qr_cr_openai_*` sehingga ter-skip runner utama. Bagian B menjalankan **full RAGAS (4 metrik)** untuk keduanya dan menulis `qr_phase2_custom.json` & `qr_cr_phase2_custom.json`.

> Pastikan sel **Setup** (bagian 1 & 2) sudah dijalankan agar `openai_generate` & helper tersedia. Sel ini juga me-*reload* `ragas_evaluator` agar perbaikan bug ikut terpakai.

In [7]:
# =====================================================================
# Reload ragas_evaluator agar perbaikan bug 'reference' ikut terpakai
# =====================================================================
import importlib
import ragas_evaluator
importlib.reload(ragas_evaluator)
from ragas_evaluator import evaluate_sample, compute_context_recall


def _extract_contexts(sample):
    """Ambil list teks konteks dari sebuah sampel phase1."""
    ctx = sample.get('contexts', []) or sample.get('retrieved_contexts', [])
    if isinstance(ctx, list) and ctx and isinstance(ctx[0], dict):
        ctx = [c.get('text', '') for c in ctx]
    return ctx


def _extract_reference(sample):
    """Prioritas: reference -> long_answer -> ground_truth (sesuai evaluator yang sudah diperbaiki)."""
    return (
        sample.get('reference', '')
        or sample.get('long_answer', '')
        or sample.get('ground_truth', '')
    )


# =====================================================================
# BAGIAN A: Re-run HANYA context_recall untuk 4 konfigurasi Kelompok 2
# (patch ke berkas phase2 yang sudah ada; metrik lain tidak disentuh)
# =====================================================================
KEL2 = '20_hybrid_tanpa_ekspansi'
KEL2_CONFIGS = ['baseline', 'qr', 'cr', 'qr_cr']

print('=' * 70)
print('BAGIAN A: Recompute context_recall untuk Kelompok 2')
print('=' * 70)

reset_api_counter()
a_start = time.time()

for cfg in KEL2_CONFIGS:
    phase1_path = RESULTS_BASE / KEL2 / f'{cfg}_phase1_answers.json'
    phase2_path = RESULTS_BASE / KEL2 / f'{cfg}_phase2_custom.json'
    if not phase1_path.exists() or not phase2_path.exists():
        print(f'[SKIP] {cfg}: berkas phase1/phase2 tidak lengkap')
        continue

    p1 = json.load(open(phase1_path, encoding='utf-8'))
    samples = p1.get('results', p1.get('answers', []))
    by_idx = {s.get('idx'): s for s in samples}

    p2 = json.load(open(phase2_path, encoding='utf-8'))
    rows = p2.get('results', [])

    print(f'\n[{cfg}] {len(rows)} baris phase2; menghitung ulang context_recall...')
    pbar = tqdm(rows, desc=f'{cfg:<12} CRec', unit='sample', ncols=110)
    n_patched = 0
    for row in pbar:
        s = by_idx.get(row.get('idx'))
        if s is None:
            continue
        reference = _extract_reference(s)
        contexts = _extract_contexts(s)
        try:
            row['context_recall'] = compute_context_recall(openai_generate, reference, contexts)
            n_patched += 1
        except Exception as e:
            print(f"  [ERROR] idx={row.get('idx')}: {type(e).__name__}: {str(e)[:60]}")
        if n_patched % 25 == 0:
            json.dump(p2, open(phase2_path, 'w', encoding='utf-8'), indent=2)
    pbar.close()

    json.dump(p2, open(phase2_path, 'w', encoding='utf-8'), indent=2)
    new_crec = mean([r['context_recall'] for r in rows if isinstance(r.get('context_recall'), (int, float))])
    print(f'[{cfg}] selesai. context_recall baru (rata-rata) = {new_crec:.4f}')

print(f'\nBagian A selesai dalam {format_eta(time.time() - a_start)} | API calls: {get_api_count()}')


# =====================================================================
# BAGIAN B: Full RAGAS (4 metrik) untuk Kelompok 3 QR & QR+CR
# phase1 bernama sh_qr_openai_* / sh_qr_cr_openai_*
# output ditulis sebagai qr_phase2_custom.json / qr_cr_phase2_custom.json
# =====================================================================
KEL3 = '30_hybrid_dengan_ekspansi'
# (config_name_output, phase1_filename)
KEL3_TARGETS = [
    ('qr',    'sh_qr_openai_phase1_answers.json'),
    ('qr_cr', 'sh_qr_cr_openai_phase1_answers.json'),
]

print('\n' + '=' * 70)
print('BAGIAN B: Full RAGAS untuk Kelompok 3 (QR & QR+CR)')
print('=' * 70)

reset_api_counter()
b_start = time.time()

for cfg, p1_name in KEL3_TARGETS:
    phase1_path = RESULTS_BASE / KEL3 / p1_name
    phase2_path = RESULTS_BASE / KEL3 / f'{cfg}_phase2_custom.json'
    if not phase1_path.exists():
        print(f'[SKIP] {cfg}: {p1_name} tidak ditemukan')
        continue

    p1 = json.load(open(phase1_path, encoding='utf-8'))
    samples = p1.get('results', p1.get('answers', []))[:500]

    # Resume-aware
    if phase2_path.exists():
        p2 = json.load(open(phase2_path, encoding='utf-8'))
        results = p2.get('results', [])
        done = {r['idx'] for r in results if all(m in r for m in REQUIRED_METRICS)}
    else:
        results, done = [], set()

    todo = [s for s in samples if s.get('idx') not in done]
    print(f'\n[{cfg}] phase1={len(samples)} | sudah={len(done)} | perlu eval={len(todo)}')
    if not todo:
        print(f'[{cfg}] semua sampel sudah dievaluasi.')
        continue

    pbar = tqdm(todo, desc=f'{cfg:<12} RAGAS', unit='sample', ncols=110)
    errors = 0
    for k, sample in enumerate(pbar, 1):
        try:
            metrics = evaluate_sample(openai_generate, sample)
        except Exception as e:
            errors += 1
            print(f"\n  [ERROR] idx={sample.get('idx')}: {type(e).__name__}: {str(e)[:60]}")
            continue
        results.append({
            'idx': sample.get('idx'),
            'ground_truth': sample.get('ground_truth') or sample.get('final_decision'),
            'predicted_label': sample.get('predicted_label'),
            'is_correct': sample.get('is_correct'),
            **metrics,
        })
        if k % 25 == 0 or k == len(todo):
            json.dump({
                'kelompok': KEL3, 'config': cfg, 'eval_model': EVAL_MODEL,
                'metrics': REQUIRED_METRICS, 'results': results,
            }, open(phase2_path, 'w', encoding='utf-8'), indent=2)
    pbar.close()

    def _safe(key):
        vals = [r.get(key) for r in results if isinstance(r.get(key), (int, float))]
        return mean(vals) if vals else 0.0
    print(f'[{cfg}] selesai. Faith={_safe("faithfulness"):.4f} '
          f'CRec={_safe("context_recall"):.4f} ARel={_safe("answer_relevancy"):.4f} '
          f'CPrec={_safe("context_precision"):.4f} | errors={errors}')
    print(f'[{cfg}] disimpan ke {phase2_path.name}')

print(f'\nBagian B selesai dalam {format_eta(time.time() - b_start)} | API calls: {get_api_count()}')
print('\n[SELESAI] Semua perbaikan diterapkan. Jalankan ulang sel ringkasan (bagian 5) untuk verifikasi.')

BAGIAN A: Recompute context_recall untuk Kelompok 2

[baseline] 500 baris phase2; menghitung ulang context_recall...


baseline     CRec:   0%|                                                          | 0/500 [00:00<?, ?sample/s]

[baseline] selesai. context_recall baru (rata-rata) = 0.8514

[qr] 500 baris phase2; menghitung ulang context_recall...


qr           CRec:   0%|                                                          | 0/500 [00:00<?, ?sample/s]

[qr] selesai. context_recall baru (rata-rata) = 0.8577

[cr] 500 baris phase2; menghitung ulang context_recall...


cr           CRec:   0%|                                                          | 0/500 [00:00<?, ?sample/s]

KeyboardInterrupt: 

In [11]:
# =====================================================================
# BAGIAN C: Auto-deteksi konfigurasi yang BELUM lengkap, lalu run
#           HANYA yang belum dievaluasi (resume-aware).
# Aman dijalankan berkali-kali: yang sudah lengkap otomatis di-skip.
# =====================================================================
import importlib
import ragas_evaluator
importlib.reload(ragas_evaluator)
from ragas_evaluator import evaluate_sample

# Throttle tqdm agar tidak melebihi iopub_msg_rate_limit Jupyter.
TQDM_KW = dict(unit='sample', ncols=110, mininterval=1.0, miniters=20)

# Seluruh 12 konfigurasi (kelompok, config). Nama berkas phase1 di-resolve otomatis.
ALL_CONFIGS = [
    ('10_bm25', c) for c in ['baseline', 'qr', 'cr', 'qr_cr']
] + [
    ('20_hybrid_tanpa_ekspansi', c) for c in ['baseline', 'qr', 'cr', 'qr_cr']
] + [
    ('30_hybrid_dengan_ekspansi', c) for c in ['baseline', 'qr', 'cr', 'qr_cr']
]


def _resolve_phase1(folder, cfg):
    """Cari berkas phase1; dukung variasi nama sh_<cfg>_openai_*."""
    d = RESULTS_BASE / folder
    for cand in [d / f'{cfg}_phase1_answers.json',
                 d / f'sh_{cfg}_openai_phase1_answers.json',
                 d / f'{cfg}_openai_phase1_answers.json']:
        if cand.exists():
            return cand
    return None


def _ragas_contexts(sample):
    ctx = sample.get('contexts', []) or sample.get('retrieved_contexts', [])
    if isinstance(ctx, list) and ctx and isinstance(ctx[0], dict):
        ctx = [c.get('text', '') for c in ctx]
    return ctx


# --- Langkah 1: deteksi konfigurasi yang belum lengkap -------------------
pending = []
print('Audit kelengkapan 12 konfigurasi:')
for folder, cfg in ALL_CONFIGS:
    p1 = _resolve_phase1(folder, cfg)
    p2 = RESULTS_BASE / folder / f'{cfg}_phase2_custom.json'
    n1 = len(json.load(open(p1, encoding='utf-8')).get('results', [])) if p1 else 0
    done = 0
    if p2.exists():
        rows = json.load(open(p2, encoding='utf-8')).get('results', [])
        done = sum(1 for r in rows if all(m in r for m in REQUIRED_METRICS))
    status = 'OK' if (n1 > 0 and done >= n1) else (f'PERLU EVAL ({done}/{n1})' if p1 else 'NO PHASE1')
    print(f'  {folder}/{cfg:<10} {status}')
    if p1 and done < n1:
        pending.append((folder, cfg, p1, p2))

print(f'\nKonfigurasi perlu di-run: {len(pending)}')
for folder, cfg, _, _ in pending:
    print(f'  - {folder}/{cfg}')

# --- Langkah 2: run hanya yang pending -----------------------------------
reset_api_counter()
c_start = time.time()

for folder, cfg, p1_path, p2_path in pending:
    samples = json.load(open(p1_path, encoding='utf-8')).get('results', [])[:500]

    # Resume: muat hasil yang sudah ada
    if p2_path.exists():
        results = json.load(open(p2_path, encoding='utf-8')).get('results', [])
        done_idx = {r['idx'] for r in results if all(m in r for m in REQUIRED_METRICS)}
    else:
        results, done_idx = [], set()

    todo = [s for s in samples if s.get('idx') not in done_idx]
    print(f'\n{"="*70}\n{folder}/{cfg}  | total={len(samples)} sudah={len(done_idx)} perlu={len(todo)}\n{"="*70}')
    if not todo:
        print('  [DONE] sudah lengkap.')
        continue

    pbar = tqdm(todo, desc=f'{cfg:<12} RAGAS', **TQDM_KW)
    errors = 0
    for k, sample in enumerate(pbar, 1):
        try:
            metrics = evaluate_sample(openai_generate, sample)
        except Exception as e:
            errors += 1
            print(f"\n  [ERROR] idx={sample.get('idx')}: {type(e).__name__}: {str(e)[:60]}")
            continue
        results.append({
            'idx': sample.get('idx'),
            'ground_truth': sample.get('ground_truth') or sample.get('final_decision'),
            'predicted_label': sample.get('predicted_label'),
            'is_correct': sample.get('is_correct'),
            **metrics,
        })
        if k % 25 == 0 or k == len(todo):
            json.dump({
                'kelompok': folder, 'config': cfg, 'eval_model': EVAL_MODEL,
                'metrics': REQUIRED_METRICS, 'results': results,
            }, open(p2_path, 'w', encoding='utf-8'), indent=2)
    pbar.close()

    def _safe(key, rows=results):
        vals = [r.get(key) for r in rows if isinstance(r.get(key), (int, float))]
        return mean(vals) if vals else 0.0
    print(f'  selesai. Faith={_safe("faithfulness"):.4f} CRec={_safe("context_recall"):.4f} '
          f'ARel={_safe("answer_relevancy"):.4f} CPrec={_safe("context_precision"):.4f} | errors={errors}')
    print(f'  disimpan ke {p2_path.name}')

print(f'\n[SELESAI] Bagian C selesai dalam {format_eta(time.time() - c_start)} | API calls: {get_api_count()}')
if not pending:
    print('Tidak ada yang perlu di-run; seluruh 12 konfigurasi sudah lengkap.')

Audit kelengkapan 12 konfigurasi:
  10_bm25/baseline   OK
  10_bm25/qr         OK
  10_bm25/cr         OK
  10_bm25/qr_cr      OK
  20_hybrid_tanpa_ekspansi/baseline   OK
  20_hybrid_tanpa_ekspansi/qr         OK
  20_hybrid_tanpa_ekspansi/cr         OK
  20_hybrid_tanpa_ekspansi/qr_cr      OK
  30_hybrid_dengan_ekspansi/baseline   OK
  30_hybrid_dengan_ekspansi/qr         OK
  30_hybrid_dengan_ekspansi/cr         OK
  30_hybrid_dengan_ekspansi/qr_cr      PERLU EVAL (0/500)

Konfigurasi perlu di-run: 1
  - 30_hybrid_dengan_ekspansi/qr_cr

30_hybrid_dengan_ekspansi/qr_cr  | total=500 sudah=0 perlu=500


qr_cr        RAGAS:   0%|                                                         | 0/500 [00:00<?, ?sample/s]

  selesai. Faith=0.9835 CRec=0.8793 ARel=0.9790 CPrec=0.5008 | errors=0
  disimpan ke qr_cr_phase2_custom.json

[SELESAI] Bagian C selesai dalam 1.2h | API calls: 5997
